# Quickstart 2 &ndash; Power Flow

*Adapted from: [PyPSA Quickstart 2 &ndash; Power Flow](https://docs.pypsa.org/latest/examples/example-2/)*

In [1]:
import typsa
from typsa.components import Bus, Carrier, CustomLineParameters, Generator, Line, Load

## Defining the Network

In [2]:
network = typsa.Network()

v_nom = 11
network.add_components(Carrier(name="AC"))
network.add_components(
    zone_1 := Bus(name="zone_1", v_nom=v_nom),
    zone_2 := Bus(name="zone_2", v_nom=v_nom),
    zone_3 := Bus(name="zone_3", v_nom=v_nom),
)
network.add_components(
    Load(name="load_1", bus=zone_1.name, p_set=50),
    Load(name="load_2", bus=zone_2.name, p_set=60),
    Load(name="load_3", bus=zone_3.name, p_set=300),
)
network.add_components(
    Generator(name="gen_a", bus=zone_1.name, p_nom=140, marginal_cost=7.5),
    Generator(name="gen_b", bus=zone_1.name, p_nom=285, marginal_cost=6),
    Generator(name="gen_c", bus=zone_2.name, p_nom=90, marginal_cost=14),
    Generator(name="gen_d", bus=zone_3.name, p_nom=85, marginal_cost=10),
)
network.add_components(
    Line(
        name="line_1",
        bus0=zone_1.name,
        bus1=zone_2.name,
        s_nom=126,
        parameters=CustomLineParameters(x=0.02, r=0.01),
    ),
    Line(
        name="line_2",
        bus0=zone_1.name,
        bus1=zone_3.name,
        s_nom=250,
        parameters=CustomLineParameters(x=0.02, r=0.01),
    ),
    Line(
        name="line_3",
        bus0=zone_2.name,
        bus1=zone_3.name,
        s_nom=130,
        parameters=CustomLineParameters(x=0.01, r=0.01),
    ),
)

## Optimizing the Network

In [ ]:
optimized_network, _ = network.optimize()

INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.02s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 7 primals, 18 duals
Objective: 2.84e+03
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Line-fix-s-lower, Line-fix-s-upper, Kirchhoff-Voltage-Law were not assigned to the network.


Running HiGHS 1.12.0 (git hash: 755a8e0): Copyright (c) 2025 HiGHS under MIT licence terms
LP linopy-problem-agbgh_0q has 18 rows; 7 cols; 27 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 2e+01]
  Cost    [6e+00, 1e+01]
  Bound   [0e+00, 0e+00]
  RHS     [5e+01, 3e+02]
Presolving model
4 rows, 7 cols, 13 nonzeros  0s
3 rows, 6 cols, 10 nonzeros  0s
Dependent equations search running on 3 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
3 rows, 6 cols, 10 nonzeros  0s
Presolve reductions: rows 3(-15); columns 6(-1); nonzeros 10(-17) 
Solving the presolved LP
Using EKK dual simplex solver - serial
  Iteration        Objective     Infeasibilities num(sum)
          0    -4.4457873746e-05 Pr: 3(452) 0s
          4     2.8350000000e+03 Pr: 0(0) 0s

Performed postsolve
Solving the original LP from the solution after postsolve

Model name          : linopy-problem-agbgh_0q
Model status        : Optimal
Simplex   iterat

In [4]:
print(optimized_network.dynamic_results.of_all_buses.marginal_price)

name      zone_1  zone_2  zone_3
snapshot                        
now          7.5   11.25    10.0


In [5]:
print(optimized_network.dynamic_results.of_all_generators.p)

name      gen_a  gen_b  gen_c  gen_d
snapshot                            
now        50.0  285.0   -0.0   75.0


In [6]:
print(optimized_network.dynamic_results.of_all_lines.p0)

name      line_1  line_2  line_3
snapshot                        
now        126.0   159.0    66.0


## Simulating the Network (Power Flow)

In [7]:
pf_results = optimized_network.simulation().pf()

INFO:pypsa.network.power_flow:Performing non-linear load-flow on AC sub-network <pypsa.networks.SubNetwork object at 0x12dbec910> for snapshots Index(['now'], dtype='str', name='snapshot')


In [8]:
print(pf_results.of_all_generators.p)

name          gen_a  gen_b  gen_c  gen_d
snapshot                                
now       53.860705  285.0   -0.0   75.0


In [9]:
print(pf_results.of_all_generators.q)

name         gen_a  gen_b  gen_c  gen_d
snapshot                               
now       7.379326    0.0    0.0    0.0


In [10]:
print(pf_results.of_all_lines.q0)

name        line_1    line_2    line_3
snapshot                              
now      -1.811567  9.190894 -4.388279
